# Planner Independent Evaluation

Evaluate the real `planning.plan_trip()` implementation using an
independent evaluator.

## Objectives

- Verify planning capacity invariants.
- Verify day coverage and place ID uniqueness.
- Verify warning behavior.
- Test determinism.
- Cover valid and invalid input cases.
- Keep the evaluator independent from the planner's internal algorithm.

## Scope

The evaluator checks the planner contract without reimplementing
ranking, geographic grouping, or ordering logic.

In [2]:
import json
import os
import shutil
from pathlib import Path

In [3]:
from pathlib import Path
import sys

# Find the repository root from the notebook location.
repo_root = Path.cwd().resolve()

while repo_root.name != "gr6_project_git" and repo_root.parent != repo_root:
    repo_root = repo_root.parent

planning_path = repo_root / "ai-ml" / "planning"

if not planning_path.exists():
    raise FileNotFoundError(f"Planning package not found: {planning_path}")

ai_ml_path = repo_root / "ai-ml"

if str(ai_ml_path) not in sys.path:
    sys.path.insert(0, str(ai_ml_path))

from planning.planner import plan_trip, PlannerInputError
from planning.models import (
    PlaceCandidate,
    PlanningDay,
    PlanningResult,
    PlanningWarning,
)

print("Real planner imported successfully")
print("Planning path:", planning_path)

Real planner imported successfully
Planning path: C:\Users\balsa\Downloads\gr6_project_git\ai-ml\planning


In [4]:
from pathlib import Path

# Find the repository root
repo_root = Path.cwd().resolve()

while repo_root.name != "gr6_project_git" and repo_root.parent != repo_root:
    repo_root = repo_root.parent

# Define project paths
ai_ml_path = repo_root / "ai-ml"
evaluation_path = ai_ml_path / "evaluation"
fixtures_path = evaluation_path / "fixtures"

# Check paths
print("Repository root:", repo_root)
print("Evaluation path:", evaluation_path)
print("Fixtures path:", fixtures_path)

if not fixtures_path.exists():
    raise FileNotFoundError(
        f"Fixtures directory not found: {fixtures_path}"
    )

print("\nFixture files:")
for path in sorted(fixtures_path.glob("*.json")):
    print("-", path.name)

Repository root: C:\Users\balsa\Downloads\gr6_project_git
Evaluation path: C:\Users\balsa\Downloads\gr6_project_git\ai-ml\evaluation
Fixtures path: C:\Users\balsa\Downloads\gr6_project_git\ai-ml\evaluation\fixtures

Fixture files:
- empty_D3_E0.json
- INVALID_D3_E5.json
- no_interests_D3_E8.json
- normal_D3_E8.json
- partial_D3_E2.json
- selection_limit_D3_E10.json


In [5]:
from pathlib import Path

evaluation_path = repo_root / "ai-ml" / "evaluation"
fixtures_path = evaluation_path / "fixtures"

if not fixtures_path.exists():
    raise FileNotFoundError(f"Fixtures directory not found: {fixtures_path}")

print("Fixtures path:", fixtures_path)
print("Fixture files:")

for path in sorted(fixtures_path.glob("*.json")):
    print("-", path.name)

Fixtures path: C:\Users\balsa\Downloads\gr6_project_git\ai-ml\evaluation\fixtures
Fixture files:
- empty_D3_E0.json
- INVALID_D3_E5.json
- no_interests_D3_E8.json
- normal_D3_E8.json
- partial_D3_E2.json
- selection_limit_D3_E10.json


In [6]:
fixtures_dir = repo_root / "ai-ml" / "evaluation" / "fixtures"

fixture_files = sorted(fixtures_dir.glob("*.json"))

print("Found fixtures:")

for path in fixture_files:
    print("-", path.name)

Found fixtures:
- empty_D3_E0.json
- INVALID_D3_E5.json
- no_interests_D3_E8.json
- normal_D3_E8.json
- partial_D3_E2.json
- selection_limit_D3_E10.json


In [7]:
import json

def load_fixture(path: Path) -> dict:
    """Load one planning fixture from JSON."""

    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


fixtures = {
    path.name: load_fixture(path)
    for path in fixture_files
}

print("Loaded fixtures:")

for filename, case in fixtures.items():
    print(f"- {filename}: {case['_case']}")

Loaded fixtures:
- empty_D3_E0.json: Empty
- INVALID_D3_E5.json: INVALID (must fail validation)
- no_interests_D3_E8.json: No-interests
- normal_D3_E8.json: Normal
- partial_D3_E2.json: Partial
- selection_limit_D3_E10.json: Selection limit


In [8]:
def build_candidates(candidate_data: list[dict]) -> list[PlaceCandidate]:
    """Convert fixture dictionaries into PlaceCandidate objects."""
    return [
        PlaceCandidate.from_dict(candidate)
        for candidate in candidate_data
    ]

print("Candidate builder is ready.")

Candidate builder is ready.


In [9]:
def expected_capacities(
    days: int,
    num_candidates: int
) -> list[int]:
    """
    Independently calculate the expected number
    of places per day.
    """
    n = min(num_candidates, 3 * days)
    q, r = divmod(n, days)

    return [q + 1] * r + [q] * (days - r)


print("Capacity helper is ready.")

Capacity helper is ready.


In [10]:
capacity_tests = [
    ("Normal", 3, 8),
    ("Selection limit", 3, 10),
    ("Partial", 3, 2),
    ("Empty", 3, 0),
]

for case_name, days, num_candidates in capacity_tests:
    capacities = expected_capacities(
        days,
        num_candidates
    )

    print(
        f"{case_name}: "
        f"D={days}, E={num_candidates} "
        f"-> {capacities}"
    )

Normal: D=3, E=8 -> [3, 3, 2]
Selection limit: D=3, E=10 -> [3, 3, 3]
Partial: D=3, E=2 -> [1, 1, 0]
Empty: D=3, E=0 -> [0, 0, 0]


In [11]:
def validate_planning_output(
    request: dict,
    response: dict
) -> list[str]:
    """
    Independently validate structural invariants
    of a planner response.
    """
    errors = []

    days = request["days"]
    candidates = request["candidatePlaces"]

    candidate_ids = {
        candidate["id"]
        for candidate in candidates
    }

    expected = expected_capacities(
        days,
        len(candidates)
    )

    # Response structure
    if not isinstance(response, dict):
        return ["Response must be a dictionary"]

    if "days" not in response:
        errors.append("Response is missing 'days'")
        return errors

    if "warnings" not in response:
        errors.append("Response is missing 'warnings'")

    result_days = response["days"]

    if not isinstance(result_days, list):
        return ["Response 'days' must be a list"]

    # Number of days
    if len(result_days) != days:
        errors.append(
            f"Expected {days} days, "
            f"got {len(result_days)}"
        )

    # Day numbers: must be exactly 1..D in order
    actual_day_numbers = [
        day.get("day")
        for day in result_days
    ]

    expected_day_numbers = list(range(1, days + 1))

    if actual_day_numbers != expected_day_numbers:
        errors.append(
            f"Expected day numbers {expected_day_numbers}, "
            f"got {actual_day_numbers}"
        )

    # Capacity per day
    for index, day in enumerate(result_days):
        if index >= days:
            break

        place_ids = day.get("placeIds", [])

        if not isinstance(place_ids, list):
            errors.append(
                f"Day {index + 1}: "
                f"'placeIds' must be a list"
            )
            continue

        actual_count = len(place_ids)
        expected_count = expected[index]

        if actual_count != expected_count:
            errors.append(
                f"Day {index + 1}: "
                f"expected {expected_count} places, "
                f"got {actual_count}"
            )

        if actual_count > 3:
            errors.append(
                f"Day {index + 1}: "
                f"maximum is 3 places, got {actual_count}"
            )

    # Collect selected IDs
    selected_ids = []

    for day in result_days:
        place_ids = day.get("placeIds", [])

        if isinstance(place_ids, list):
            selected_ids.extend(place_ids)

    # Total selected places
    expected_total = sum(expected)

    if len(selected_ids) != expected_total:
        errors.append(
            f"Expected {expected_total} selected places, "
            f"got {len(selected_ids)}"
        )

    # Duplicate IDs
    if len(selected_ids) != len(set(selected_ids)):
        errors.append(
            "Duplicate place IDs found in response"
        )

    # Unknown IDs
    invalid_ids = set(selected_ids) - candidate_ids

    if invalid_ids:
        errors.append(
            f"Unknown place IDs found: "
            f"{sorted(invalid_ids)}"
        )

    return errors

In [12]:
normal_case = fixtures["normal_D3_E8.json"]

request = normal_case["request"]

candidates = build_candidates(
    request["candidatePlaces"]
)

result = plan_trip(
    destination_id=request["destinationId"],
    days=request["days"],
    interests=request["interests"],
    candidate_places=candidates,
)

actual_response = result.to_dict()

print("Planner output:")
print(json.dumps(
    actual_response,
    indent=2,
    ensure_ascii=False
))

Planner output:
{
  "days": [
    {
      "day": 1,
      "placeIds": [
        "p1",
        "p3",
        "p2"
      ]
    },
    {
      "day": 2,
      "placeIds": [
        "p4",
        "p7",
        "p5"
      ]
    },
    {
      "day": 3,
      "placeIds": [
        "p6",
        "p8"
      ]
    }
  ],
  "warnings": []
}


In [13]:
errors = validate_planning_output(
    request,
    actual_response
)

if not errors:
    print("PASS - Normal case")
else:
    print("FAIL - Normal case")

    for error in errors:
        print("-", error)

PASS - Normal case


In [14]:
def validate_warning_behavior(
    request: dict,
    response: dict
) -> list[str]:
    """
    Independently validate warning behavior
    according to the planner coverage contract.
    """
    errors = []

    days = request["days"]
    num_candidates = len(
        request["candidatePlaces"]
    )

    n = min(num_candidates, 3 * days)

    warnings = response.get("warnings", [])

    warning_codes = {
        warning.get("code")
        for warning in warnings
        if isinstance(warning, dict)
    }

    coverage_warnings = {
        "NO_PLACES_AVAILABLE",
        "PARTIAL_ITINERARY",
    }

    # N = 0
    if n == 0:
        if "NO_PLACES_AVAILABLE" not in warning_codes:
            errors.append(
                "Expected NO_PLACES_AVAILABLE warning "
                "when N = 0"
            )

    # 0 < N < D
    elif 0 < n < days:
        if "PARTIAL_ITINERARY" not in warning_codes:
            errors.append(
                "Expected PARTIAL_ITINERARY warning "
                "when 0 < N < D"
            )

    # N >= D
    else:
        unexpected = warning_codes & coverage_warnings

        if unexpected:
            errors.append(
                "N >= D must not contain coverage warnings: "
                f"{sorted(unexpected)}"
            )

    return errors

In [15]:
warning_tests = [
    "empty_D3_E0.json",
    "partial_D3_E2.json",
]

for filename in warning_tests:
    case = fixtures[filename]
    request = case["request"]

    candidates = build_candidates(
        request["candidatePlaces"]
    )

    result = plan_trip(
        destination_id=request["destinationId"],
        days=request["days"],
        interests=request["interests"],
        candidate_places=candidates,
    )

    response = result.to_dict()

    errors = validate_warning_behavior(
        request,
        response
    )

    if not errors:
        print(f"PASS - {case['_case']}")
    else:
        print(f"FAIL - {case['_case']}")
        for error in errors:
            print(" -", error)

    print("Warnings:", response["warnings"])
    print()

PASS - Empty
Warnings: [{'code': 'NO_PLACES_AVAILABLE', 'message': 'No suitable places were found in the current candidate pool.'}]

PASS - Partial
Warnings: [{'code': 'PARTIAL_ITINERARY', 'message': 'The current candidate pool does not contain enough places to cover every requested day.'}]



In [16]:
valid_case_files = [
    "empty_D3_E0.json",
    "no_interests_D3_E8.json",
    "normal_D3_E8.json",
    "partial_D3_E2.json",
    "selection_limit_D3_E10.json",
]

valid_results = {}

for filename in valid_case_files:
    case = fixtures[filename]
    request = case["request"]

    candidates = build_candidates(
        request["candidatePlaces"]
    )

    result = plan_trip(
        destination_id=request["destinationId"],
        days=request["days"],
        interests=request["interests"],
        candidate_places=candidates,
    )

    response = result.to_dict()

    errors = []

    errors.extend(
        validate_planning_output(
            request,
            response
        )
    )

    errors.extend(
        validate_warning_behavior(
            request,
            response
        )
    )

    valid_results[filename] = {
        "case": case["_case"],
        "response": response,
        "errors": errors,
    }

    status = "PASS" if not errors else "FAIL"

    print(f"{status} - {case['_case']}")

    for error in errors:
        print("  -", error)

PASS - Empty
PASS - No-interests
PASS - Normal
PASS - Partial
PASS - Selection limit


## INVALID Fixture Validation

The INVALID fixture contains an intentionally malformed planner response.

The evaluator validates this response independently against the planning
contract and must reject it when structural violations are detected.

In [17]:
invalid_case = fixtures["INVALID_D3_E5.json"]

invalid_request = invalid_case["request"]
invalid_response = invalid_case["response"]

errors = validate_planning_output(
    invalid_request,
    invalid_response
)

if errors:
    print("PASS - INVALID case correctly rejected")
    print(f"Detected {len(errors)} violations:")

    for error in errors:
        print("  -", error)
else:
    print("FAIL - INVALID case was incorrectly accepted")

PASS - INVALID case correctly rejected
Detected 5 violations:
  - Expected 3 days, got 2
  - Expected day numbers [1, 2, 3], got [1, 2]
  - Expected 5 selected places, got 4
  - Duplicate place IDs found in response
  - Unknown place IDs found: ['p99']


In [18]:
# Prepare a valid baseline response independently

baseline_case = fixtures["normal_D3_E8.json"]
baseline_request = baseline_case["request"]

baseline_candidates = build_candidates(
    baseline_request["candidatePlaces"]
)

baseline_result = plan_trip(
    destination_id=baseline_request["destinationId"],
    days=baseline_request["days"],
    interests=baseline_request["interests"],
    candidate_places=baseline_candidates,
)

baseline_response = baseline_result.to_dict()

print("Baseline response prepared.")
print("Days:", len(baseline_response["days"]))
print("Warnings:", baseline_response["warnings"])

Baseline response prepared.
Days: 3
Warnings: []


In [19]:
import copy

invalid_planning_cases = {}

# 1. Unknown selected ID
case = copy.deepcopy(baseline_response)
case["days"][0]["placeIds"][0] = "unknown_place"
invalid_planning_cases["Unknown selected ID"] = case


# 2. Duplicate selected ID
case = copy.deepcopy(baseline_response)
first_id = case["days"][0]["placeIds"][0]
case["days"][0]["placeIds"][1] = first_id
invalid_planning_cases["Duplicate selected ID"] = case


# 3. Missing day
case = copy.deepcopy(baseline_response)
case["days"].pop()
invalid_planning_cases["Missing day"] = case


print("Prepared invalid PlanningResult cases:")
for name in invalid_planning_cases:
    print("-", name)

Prepared invalid PlanningResult cases:
- Unknown selected ID
- Duplicate selected ID
- Missing day


In [20]:
# 4. Extra day
case = copy.deepcopy(baseline_response)

extra_day = {
    "day": 4,
    "placeIds": []
}

case["days"].append(extra_day)
invalid_planning_cases["Extra day"] = case


# 5. Wrong capacity
case = copy.deepcopy(baseline_response)

# Remove one place from Day 1
case["days"][0]["placeIds"].pop()

invalid_planning_cases["Wrong capacity"] = case


# 6. Wrong total selected count
case = copy.deepcopy(baseline_response)

# Add an extra selected ID
case["days"][0]["placeIds"].append(
    "extra_place"
)

invalid_planning_cases["Wrong total selected count"] = case


print("Updated invalid PlanningResult cases:")
for name in invalid_planning_cases:
    print("-", name)

Updated invalid PlanningResult cases:
- Unknown selected ID
- Duplicate selected ID
- Missing day
- Extra day
- Wrong capacity
- Wrong total selected count


In [21]:
# 7. Wrong warning

warning_case = fixtures["empty_D3_E0.json"]
warning_request = warning_case["request"]

warning_candidates = build_candidates(
    warning_request["candidatePlaces"]
)

warning_result = plan_trip(
    destination_id=warning_request["destinationId"],
    days=warning_request["days"],
    interests=warning_request["interests"],
    candidate_places=warning_candidates,
)

wrong_warning_response = warning_result.to_dict()

# Remove the warning that should exist for an empty case
wrong_warning_response["warnings"] = []

invalid_planning_cases["Wrong warning"] = wrong_warning_response

print("All deliberately invalid PlanningResult cases prepared:")
for name in invalid_planning_cases:
    print("-", name)

All deliberately invalid PlanningResult cases prepared:
- Unknown selected ID
- Duplicate selected ID
- Missing day
- Extra day
- Wrong capacity
- Wrong total selected count
- Wrong warning


In [22]:
# Prove that the independent checker rejects all deliberately invalid outputs

invalid_requests = {
    "Unknown selected ID": baseline_request,
    "Duplicate selected ID": baseline_request,
    "Missing day": baseline_request,
    "Extra day": baseline_request,
    "Wrong capacity": baseline_request,
    "Wrong total selected count": baseline_request,
    "Wrong warning": warning_request,
}

invalid_check_results = {}

for name, invalid_response in invalid_planning_cases.items():
    request = invalid_requests[name]

    errors = validate_planning_output(
        request,
        invalid_response
    )

    # Check warning semantics for the wrong-warning case
    if name == "Wrong warning":
        warning_errors = validate_warning_behavior(
            request,
            invalid_response
        )
        errors.extend(warning_errors)

    invalid_check_results[name] = errors

    if errors:
        print(f"PASS - {name} correctly rejected")
        for error in errors:
            print(f"  - {error}")
    else:
        print(f"FAIL - {name} was incorrectly accepted")

print("\nInvalid-output checker test completed.")

PASS - Unknown selected ID correctly rejected
  - Unknown place IDs found: ['unknown_place']
PASS - Duplicate selected ID correctly rejected
  - Duplicate place IDs found in response
PASS - Missing day correctly rejected
  - Expected 3 days, got 2
  - Expected day numbers [1, 2, 3], got [1, 2]
  - Expected 8 selected places, got 6
PASS - Extra day correctly rejected
  - Expected 3 days, got 4
  - Expected day numbers [1, 2, 3], got [1, 2, 3, 4]
PASS - Wrong capacity correctly rejected
  - Day 1: expected 3 places, got 2
  - Expected 8 selected places, got 7
PASS - Wrong total selected count correctly rejected
  - Day 1: expected 3 places, got 4
  - Day 1: maximum is 3 places, got 4
  - Expected 8 selected places, got 9
  - Unknown place IDs found: ['extra_place']
PASS - Wrong warning correctly rejected
  - Expected NO_PLACES_AVAILABLE warning when N = 0

Invalid-output checker test completed.


## Determinism Check

The same valid planning request is executed twice.

The evaluator compares both outputs and expects identical results,
ensuring deterministic planner behavior for the tested fixtures.

In [23]:
def run_planner(case: dict) -> dict:
    """Run the real planner for one fixture case."""
    request = case["request"]

    candidates = build_candidates(
        request["candidatePlaces"]
    )

    result = plan_trip(
        destination_id=request["destinationId"],
        days=request["days"],
        interests=request["interests"],
        candidate_places=candidates,
    )

    return result.to_dict()


determinism_tests = [
    "normal_D3_E8.json",
    "selection_limit_D3_E10.json",
    "partial_D3_E2.json",
    "empty_D3_E0.json",
]

for filename in determinism_tests:
    case = fixtures[filename]

    first_run = run_planner(case)
    second_run = run_planner(case)

    if first_run == second_run:
        print(f"PASS - Deterministic: {case['_case']}")
    else:
        print(f"FAIL - Non-deterministic: {case['_case']}")

PASS - Deterministic: Normal
PASS - Deterministic: Selection limit
PASS - Deterministic: Partial
PASS - Deterministic: Empty


In [24]:
def validate_day_numbers(
    request: dict,
    response: dict
) -> list[str]:
    """Validate that day numbers are sequential and start from 1."""
    errors = []

    expected_days = list(
        range(1, request["days"] + 1)
    )

    actual_days = [
        day.get("day")
        for day in response.get("days", [])
    ]

    if actual_days != expected_days:
        errors.append(
            f"Expected day numbers {expected_days}, "
            f"got {actual_days}"
        )

    return errors


for filename in valid_case_files:
    case = fixtures[filename]

    response = run_planner(case)

    errors = validate_day_numbers(
        case["request"],
        response
    )

    if not errors:
        print(f"PASS - Day numbers: {case['_case']}")
    else:
        print(f"FAIL - Day numbers: {case['_case']}")

        for error in errors:
            print("  -", error)

PASS - Day numbers: Empty
PASS - Day numbers: No-interests
PASS - Day numbers: Normal
PASS - Day numbers: Partial
PASS - Day numbers: Selection limit


## Golden / Reference Checks

Golden checks are used only for small, clearly justified cases.

The evaluator does not require the exact ordering of all places unless
the fixture provides a specific behavior that should be preserved.

In [25]:
partial_case = fixtures["partial_D3_E2.json"]

partial_response = run_planner(partial_case)

selected_ids = [
    place_id
    for day in partial_response["days"]
    for place_id in day["placeIds"]
]

expected_ids = {"p1", "p2"}

errors = []

if set(selected_ids) != expected_ids:
    errors.append(
        f"Expected selected IDs {sorted(expected_ids)}, "
        f"got {sorted(set(selected_ids))}"
    )

expected_counts = [1, 1, 0]

actual_counts = [
    len(day["placeIds"])
    for day in partial_response["days"]
]

if actual_counts != expected_counts:
    errors.append(
        f"Expected daily counts {expected_counts}, "
        f"got {actual_counts}"
    )

if not errors:
    print("PASS - Golden check: Partial")
else:
    print("FAIL - Golden check: Partial")

    for error in errors:
        print("  -", error)

PASS - Golden check: Partial


In [26]:
selection_case = fixtures["selection_limit_D3_E10.json"]

selection_response = run_planner(selection_case)

selected_ids = [
    place_id
    for day in selection_response["days"]
    for place_id in day["placeIds"]
]

errors = []

# Maximum selection rule
if len(selected_ids) != 9:
    errors.append(
        f"Expected exactly 9 selected places, "
        f"got {len(selected_ids)}"
    )

# All selected IDs must come from the input candidates
candidate_ids = {
    candidate["id"]
    for candidate in selection_case["request"]["candidatePlaces"]
}

unknown_ids = set(selected_ids) - candidate_ids

if unknown_ids:
    errors.append(
        f"Unknown selected IDs: {sorted(unknown_ids)}"
    )

# No duplicates
if len(selected_ids) != len(set(selected_ids)):
    errors.append(
        "Duplicate place IDs found"
    )

# Expected daily capacity: 3 / 3 / 3
actual_counts = [
    len(day["placeIds"])
    for day in selection_response["days"]
]

if actual_counts != [3, 3, 3]:
    errors.append(
        f"Expected daily counts [3, 3, 3], "
        f"got {actual_counts}"
    )

if not errors:
    print("PASS - Golden check: Selection limit")
else:
    print("FAIL - Golden check: Selection limit")

    for error in errors:
        print("  -", error)

PASS - Golden check: Selection limit


In [27]:
normal_case = fixtures["normal_D3_E8.json"]

normal_response = run_planner(normal_case)

selected_ids = {
    place_id
    for day in normal_response["days"]
    for place_id in day["placeIds"]
}

# Reference places for the history interest
expected_relevant_ids = {
    "p1",
    "p2",
    "p3",
}

errors = []

# The history-relevant places should be selected
missing_relevant = expected_relevant_ids - selected_ids

if missing_relevant:
    errors.append(
        f"Missing expected relevant places: "
        f"{sorted(missing_relevant)}"
    )

# Total selected places must respect the 3 places/day limit
if len(selected_ids) != 8:
    errors.append(
        f"Expected 8 selected places, "
        f"got {len(selected_ids)}"
    )

if not errors:
    print("PASS - Golden check: Normal")
else:
    print("FAIL - Golden check: Normal")

    for error in errors:
        print("  -", error)

PASS - Golden check: Normal


In [28]:
# Edge Case: D=4, E=7
# Expected: N=7, q=1, r=3
# Expected capacities: [2, 2, 2, 1]

base_case = fixtures["normal_D3_E8.json"]

case_D4_E7 = {
    "request": {
        "destinationId": base_case["request"]["destinationId"],
        "days": 4,
        "interests": base_case["request"]["interests"],
        "candidatePlaces": base_case["request"]["candidatePlaces"][:7],
    }
}

response_D4_E7 = run_planner(case_D4_E7)

print("Planner output:")
print(response_D4_E7)

violations = validate_planning_output(
    case_D4_E7["request"],
    response_D4_E7
)

assert not violations, violations

actual_capacities = [
    len(day["placeIds"])
    for day in response_D4_E7["days"]
]

expected_capacities_D4_E7 = [2, 2, 2, 1]

assert actual_capacities == expected_capacities_D4_E7, (
    f"Expected {expected_capacities_D4_E7}, "
    f"got {actual_capacities}"
)

warning_codes = {
    warning["code"]
    for warning in response_D4_E7["warnings"]
}

assert "NO_PLACES_AVAILABLE" not in warning_codes
assert "PARTIAL_ITINERARY" not in warning_codes

print("\nPASS - Edge case: D=4, E=7")
print("Expected capacities:", expected_capacities_D4_E7)
print("Actual capacities:", actual_capacities)
print("Warnings:", warning_codes)

Planner output:
{'days': [{'day': 1, 'placeIds': ['p1', 'p3']}, {'day': 2, 'placeIds': ['p2', 'p6']}, {'day': 3, 'placeIds': ['p4', 'p7']}, {'day': 4, 'placeIds': ['p5']}], 'warnings': []}

PASS - Edge case: D=4, E=7
Expected capacities: [2, 2, 2, 1]
Actual capacities: [2, 2, 2, 1]
Warnings: set()


In [29]:
# Edge Case: D=4, E=2
# Expected: N=2, q=0, r=2
# Expected capacities: [1, 1, 0, 0]
# Since 0 < N < D, PARTIAL_ITINERARY is required.

base_case = fixtures["normal_D3_E8.json"]

case_D4_E2 = {
    "request": {
        "destinationId": base_case["request"]["destinationId"],
        "days": 4,
        "interests": base_case["request"]["interests"],
        "candidatePlaces": base_case["request"]["candidatePlaces"][:2],
    }
}

response_D4_E2 = run_planner(case_D4_E2)

print("Planner output:")
print(response_D4_E2)

violations = validate_planning_output(
    case_D4_E2["request"],
    response_D4_E2
)

assert not violations, violations

actual_capacities = [
    len(day["placeIds"])
    for day in response_D4_E2["days"]
]

expected_capacities_D4_E2 = [1, 1, 0, 0]

assert actual_capacities == expected_capacities_D4_E2, (
    f"Expected {expected_capacities_D4_E2}, "
    f"got {actual_capacities}"
)

warning_codes = {
    warning["code"]
    for warning in response_D4_E2["warnings"]
}

assert "PARTIAL_ITINERARY" in warning_codes

print("\nPASS - Edge case: D=4, E=2")
print("Expected capacities:", expected_capacities_D4_E2)
print("Actual capacities:", actual_capacities)
print("Warnings:", warning_codes)

Planner output:
{'days': [{'day': 1, 'placeIds': ['p1']}, {'day': 2, 'placeIds': ['p2']}, {'day': 3, 'placeIds': []}, {'day': 4, 'placeIds': []}], 'warnings': [{'code': 'PARTIAL_ITINERARY', 'message': 'The current candidate pool does not contain enough places to cover every requested day.'}]}

PASS - Edge case: D=4, E=2
Expected capacities: [1, 1, 0, 0]
Actual capacities: [1, 1, 0, 0]
Warnings: {'PARTIAL_ITINERARY'}


In [30]:
# Edge Case: D=1
# E=8 -> N=min(8, 3*1)=3
# Expected capacity: [3]
# No coverage warning because N >= D

base_case = fixtures["normal_D3_E8.json"]

case_D1 = {
    "request": {
        "destinationId": base_case["request"]["destinationId"],
        "days": 1,
        "interests": base_case["request"]["interests"],
        "candidatePlaces": base_case["request"]["candidatePlaces"],
    }
}

response_D1 = run_planner(case_D1)

print("Planner output:")
print(response_D1)

violations = validate_planning_output(
    case_D1["request"],
    response_D1
)

assert not violations, violations

actual_capacities = [
    len(day["placeIds"])
    for day in response_D1["days"]
]

assert actual_capacities == [3], (
    f"Expected [3], got {actual_capacities}"
)

warning_codes = {
    warning["code"]
    for warning in response_D1["warnings"]
}

assert "NO_PLACES_AVAILABLE" not in warning_codes
assert "PARTIAL_ITINERARY" not in warning_codes

print("\nPASS - Edge case: D=1")
print("Expected capacities: [3]")
print("Actual capacities:", actual_capacities)
print("Warnings:", warning_codes)

Planner output:
{'days': [{'day': 1, 'placeIds': ['p1', 'p3', 'p2']}], 'warnings': []}

PASS - Edge case: D=1
Expected capacities: [3]
Actual capacities: [3]
Warnings: set()


In [31]:
# Edge Case: D=14
# E=8 -> N=8
# q=0, r=8
# Expected capacities: [1,1,1,1,1,1,1,1,0,0,0,0,0,0]
# PARTIAL_ITINERARY is required because N < D

base_case = fixtures["normal_D3_E8.json"]

case_D14 = {
    "request": {
        "destinationId": base_case["request"]["destinationId"],
        "days": 14,
        "interests": base_case["request"]["interests"],
        "candidatePlaces": base_case["request"]["candidatePlaces"],
    }
}

response_D14 = run_planner(case_D14)

print("Planner output:")
print(response_D14)

violations = validate_planning_output(
    case_D14["request"],
    response_D14
)

assert not violations, violations

actual_capacities = [
    len(day["placeIds"])
    for day in response_D14["days"]
]

expected_capacities_D14 = [1] * 8 + [0] * 6

assert actual_capacities == expected_capacities_D14, (
    f"Expected {expected_capacities_D14}, "
    f"got {actual_capacities}"
)

warning_codes = {
    warning["code"]
    for warning in response_D14["warnings"]
}

assert "PARTIAL_ITINERARY" in warning_codes
assert "NO_PLACES_AVAILABLE" not in warning_codes

print("\nPASS - Edge case: D=14")
print("Expected capacities:", expected_capacities_D14)
print("Actual capacities:", actual_capacities)
print("Warnings:", warning_codes)

Planner output:
{'days': [{'day': 1, 'placeIds': ['p1']}, {'day': 2, 'placeIds': ['p2']}, {'day': 3, 'placeIds': ['p3']}, {'day': 4, 'placeIds': ['p4']}, {'day': 5, 'placeIds': ['p5']}, {'day': 6, 'placeIds': ['p6']}, {'day': 7, 'placeIds': ['p7']}, {'day': 8, 'placeIds': ['p8']}, {'day': 9, 'placeIds': []}, {'day': 10, 'placeIds': []}, {'day': 11, 'placeIds': []}, {'day': 12, 'placeIds': []}, {'day': 13, 'placeIds': []}, {'day': 14, 'placeIds': []}], 'warnings': [{'code': 'PARTIAL_ITINERARY', 'message': 'The current candidate pool does not contain enough places to cover every requested day.'}]}

PASS - Edge case: D=14
Expected capacities: [1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0]
Actual capacities: [1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0]
Warnings: {'PARTIAL_ITINERARY'}


In [32]:
# Edge Case: E > 3D
# D=3, E=10
# N=min(10, 3*3)=9
# Expected capacities: [3, 3, 3]
# One candidate must remain unselected.

base_case = fixtures["selection_limit_D3_E10.json"]

response_E_gt_3D = run_planner(base_case)

print("Planner output:")
print(response_E_gt_3D)

violations = validate_planning_output(
    base_case["request"],
    response_E_gt_3D
)

assert not violations, violations

actual_capacities = [
    len(day["placeIds"])
    for day in response_E_gt_3D["days"]
]

assert actual_capacities == [3, 3, 3], (
    f"Expected [3, 3, 3], got {actual_capacities}"
)

selected_ids = {
    place_id
    for day in response_E_gt_3D["days"]
    for place_id in day["placeIds"]
}

candidate_ids = {
    candidate["id"]
    for candidate in base_case["request"]["candidatePlaces"]
}

assert len(selected_ids) == 9
assert selected_ids.issubset(candidate_ids)
assert len(candidate_ids - selected_ids) == 1

warning_codes = {
    warning["code"]
    for warning in response_E_gt_3D["warnings"]
}

assert not warning_codes

print("\nPASS - Edge case: E > 3D")
print("Candidates (E):", len(candidate_ids))
print("Maximum capacity (3D):", 9)
print("Selected (N):", len(selected_ids))
print("Unselected:", sorted(candidate_ids - selected_ids))
print("Capacities:", actual_capacities)
print("Warnings:", warning_codes)

Planner output:
{'days': [{'day': 1, 'placeIds': ['p1', 'p3', 'p2']}, {'day': 2, 'placeIds': ['p10', 'p8', 'p4']}, {'day': 3, 'placeIds': ['p5', 'p7', 'p6']}], 'warnings': []}

PASS - Edge case: E > 3D
Candidates (E): 10
Maximum capacity (3D): 9
Selected (N): 9
Unselected: ['p9']
Capacities: [3, 3, 3]
Warnings: set()


In [33]:
# Negative Control: Invalid days
# The planner should reject days outside the valid range 1..14.

base_case = fixtures["normal_D3_E8.json"]

for invalid_days in [0, 15]:
    case = {
        "request": {
            "destinationId": base_case["request"]["destinationId"],
            "days": invalid_days,
            "interests": base_case["request"]["interests"],
            "candidatePlaces": base_case["request"]["candidatePlaces"],
        }
    }

    try:
        run_planner(case)
        print(f"FAIL - days={invalid_days} was accepted")
    except Exception as exc:
        print(f"PASS - days={invalid_days} rejected")
        print(f"  Exception: {type(exc).__name__}: {exc}")

PASS - days=0 rejected
  Exception: PlannerInputError: days must be an integer from 1 to 14
PASS - days=15 rejected
  Exception: PlannerInputError: days must be an integer from 1 to 14


In [34]:
# Negative Controls: Malformed candidates
# Invalid candidate data should be rejected by the real planner/models.

base_case = fixtures["normal_D3_E8.json"]

malformed_candidates = {
    "missing_id": {
        **base_case["request"]["candidatePlaces"][0],
        "id": None,
    },
    "invalid_category_ids": {
        **base_case["request"]["candidatePlaces"][0],
        "categoryIds": "historic_site",
    },
    "invalid_latitude": {
        **base_case["request"]["candidatePlaces"][0],
        "latitude": "not-a-number",
    },
    "nonfinite_longitude": {
        **base_case["request"]["candidatePlaces"][0],
        "longitude": float("inf"),
    },
}

for case_name, bad_candidate in malformed_candidates.items():
    request = {
        "destinationId": base_case["request"]["destinationId"],
        "days": 3,
        "interests": base_case["request"]["interests"],
        "candidatePlaces": [bad_candidate],
    }

    case = {"request": request}

    try:
        run_planner(case)
        print(f"FAIL - {case_name} was accepted")
    except Exception as exc:
        print(f"PASS - {case_name} rejected")
        print(f"  Exception: {type(exc).__name__}: {exc}")

PASS - missing_id rejected
  Exception: ValueError: id must be a string
PASS - invalid_category_ids rejected
  Exception: ValueError: categoryIds must be a list of strings
PASS - invalid_latitude rejected
  Exception: ValueError: latitude must be numeric
PASS - nonfinite_longitude rejected
  Exception: ValueError: Invalid longitude for place p1


In [35]:
# Negative Control: Destination mismatch
# A candidate from another destination should be rejected.

base_case = fixtures["normal_D3_E8.json"]

mismatched_candidate = {
    **base_case["request"]["candidatePlaces"][0],
    "destinationId": "different-destination",
}

case_destination_mismatch = {
    "request": {
        "destinationId": base_case["request"]["destinationId"],
        "days": 3,
        "interests": base_case["request"]["interests"],
        "candidatePlaces": [mismatched_candidate],
    }
}

try:
    run_planner(case_destination_mismatch)
    print("FAIL - destination mismatch was accepted")
except Exception as exc:
    print("PASS - destination mismatch rejected")
    print(f"  Exception: {type(exc).__name__}: {exc}")

PASS - destination mismatch rejected
  Exception: PlannerInputError: candidate p1 belongs to different-destination, not istanbul-tr


In [36]:
# Negative Control: Duplicate candidate IDs
# Candidate IDs must be unique within the input pool.

base_case = fixtures["normal_D3_E8.json"]

candidate_1 = base_case["request"]["candidatePlaces"][0]
candidate_2 = {
    **base_case["request"]["candidatePlaces"][1],
    "id": candidate_1["id"],
}

case_duplicate_ids = {
    "request": {
        "destinationId": base_case["request"]["destinationId"],
        "days": 3,
        "interests": base_case["request"]["interests"],
        "candidatePlaces": [candidate_1, candidate_2],
    }
}

try:
    run_planner(case_duplicate_ids)
    print("FAIL - duplicate candidate IDs were accepted")
except Exception as exc:
    print("PASS - duplicate candidate IDs rejected")
    print(f"  Exception: {type(exc).__name__}: {exc}")

PASS - duplicate candidate IDs rejected
  Exception: PlannerInputError: candidate place IDs must be unique


In [37]:
# Negative Control: Duplicate interests
# Interest IDs should not be repeated in the request.

base_case = fixtures["normal_D3_E8.json"]

case_duplicate_interests = {
    "request": {
        "destinationId": base_case["request"]["destinationId"],
        "days": 3,
        "interests": ["history", "history"],
        "candidatePlaces": base_case["request"]["candidatePlaces"],
    }
}

try:
    run_planner(case_duplicate_interests)
    print("FAIL - duplicate interests were accepted")
except Exception as exc:
    print("PASS - duplicate interests rejected")
    print(f"  Exception: {type(exc).__name__}: {exc}")

PASS - duplicate interests rejected
  Exception: PlannerInputError: interests must not contain duplicates


In [38]:
# Negative Controls: Invalid coordinate ranges
# Latitude must be between -90 and 90.
# Longitude must be between -180 and 180.

base_case = fixtures["normal_D3_E8.json"]
base_candidate = base_case["request"]["candidatePlaces"][0]

invalid_coordinates = {
    "latitude_above_range": {
        **base_candidate,
        "latitude": 91,
    },
    "latitude_below_range": {
        **base_candidate,
        "latitude": -91,
    },
    "longitude_above_range": {
        **base_candidate,
        "longitude": 181,
    },
    "longitude_below_range": {
        **base_candidate,
        "longitude": -181,
    },
}

for case_name, bad_candidate in invalid_coordinates.items():
    request = {
        "destinationId": base_case["request"]["destinationId"],
        "days": 3,
        "interests": base_case["request"]["interests"],
        "candidatePlaces": [bad_candidate],
    }

    case = {"request": request}

    try:
        run_planner(case)
        print(f"FAIL - {case_name} was accepted")
    except Exception as exc:
        print(f"PASS - {case_name} rejected")
        print(f"  Exception: {type(exc).__name__}: {exc}")

PASS - latitude_above_range rejected
  Exception: ValueError: Invalid latitude for place p1
PASS - latitude_below_range rejected
  Exception: ValueError: Invalid latitude for place p1
PASS - longitude_above_range rejected
  Exception: ValueError: Invalid longitude for place p1
PASS - longitude_below_range rejected
  Exception: ValueError: Invalid longitude for place p1


In [39]:
# Edge Case: Overlap
# Several candidates share the same categories and have nearby coordinates.
# We do not assert ranking or exact ordering.
# We only verify the planner contract and invariants.

base_case = fixtures["normal_D3_E8.json"]

overlap_candidates = []

for i, candidate in enumerate(base_case["request"]["candidatePlaces"][:5]):
    overlap_candidates.append({
        **candidate,
        "categoryIds": ["historic_site"],
        "latitude": 41.0080 + (i * 0.0001),
        "longitude": 28.9800 + (i * 0.0001),
    })

case_overlap = {
    "request": {
        "destinationId": base_case["request"]["destinationId"],
        "days": 3,
        "interests": ["history"],
        "candidatePlaces": overlap_candidates,
    }
}

response_overlap = run_planner(case_overlap)

print("Planner output:")
print(response_overlap)

violations = validate_planning_output(
    case_overlap["request"],
    response_overlap
)

assert not violations, violations

selected_ids = [
    place_id
    for day in response_overlap["days"]
    for place_id in day["placeIds"]
]

assert len(selected_ids) == 5
assert len(selected_ids) == len(set(selected_ids))

print("\nPASS - Edge case: Overlap")
print("Selected places:", selected_ids)
print("Capacities:", [
    len(day["placeIds"])
    for day in response_overlap["days"]
])

Planner output:
{'days': [{'day': 1, 'placeIds': ['p1', 'p2']}, {'day': 2, 'placeIds': ['p3', 'p4']}, {'day': 3, 'placeIds': ['p5']}], 'warnings': []}

PASS - Edge case: Overlap
Selected places: ['p1', 'p2', 'p3', 'p4', 'p5']
Capacities: [2, 2, 1]


In [40]:
# Edge Case: Tie
# Candidates have identical categories and identical coordinates.
# No exact ranking/order is asserted.

base_case = fixtures["normal_D3_E8.json"]

tie_candidates = []

for i, candidate in enumerate(base_case["request"]["candidatePlaces"][:6]):
    tie_candidates.append({
        **candidate,
        "categoryIds": ["historic_site"],
        "latitude": 41.0086,
        "longitude": 28.9802,
    })

case_tie = {
    "request": {
        "destinationId": base_case["request"]["destinationId"],
        "days": 3,
        "interests": ["history"],
        "candidatePlaces": tie_candidates,
    }
}

response_tie = run_planner(case_tie)

print("Planner output:")
print(response_tie)

violations = validate_planning_output(
    case_tie["request"],
    response_tie
)

assert not violations, violations

selected_ids = [
    place_id
    for day in response_tie["days"]
    for place_id in day["placeIds"]
]

assert len(selected_ids) == 6
assert len(selected_ids) == len(set(selected_ids))

actual_capacities = [
    len(day["placeIds"])
    for day in response_tie["days"]
]

assert actual_capacities == [2, 2, 2]

print("\nPASS - Edge case: Tie")
print("Selected places:", selected_ids)
print("Capacities:", actual_capacities)

Planner output:
{'days': [{'day': 1, 'placeIds': ['p1', 'p2']}, {'day': 2, 'placeIds': ['p3', 'p4']}, {'day': 3, 'placeIds': ['p5', 'p6']}], 'warnings': []}

PASS - Edge case: Tie
Selected places: ['p1', 'p2', 'p3', 'p4', 'p5', 'p6']
Capacities: [2, 2, 2]


## No-Interests Case

This case verifies that the planner still produces a structurally valid
itinerary when no user interests are provided.

In [41]:
no_interests_case = fixtures["no_interests_D3_E8.json"]

no_interests_response = run_planner(
    no_interests_case
)

errors = validate_planning_output(
    no_interests_case["request"],
    no_interests_response
)

errors.extend(
    validate_day_numbers(
        no_interests_case["request"],
        no_interests_response
    )
)

if not errors:
    print("PASS - No-interests contract check")

    print(
        "Selected places:",
        sum(
            len(day["placeIds"])
            for day in no_interests_response["days"]
        )
    )

    print(
        "Warnings:",
        no_interests_response["warnings"]
    )

else:
    print("FAIL - No-interests contract check")

    for error in errors:
        print("  -", error)

PASS - No-interests contract check
Selected places: 8
Warnings: []


In [42]:
# Final Evaluation Summary

evaluation_summary = [
    ("Valid fixtures", "5/5 PASS"),
    ("INVALID fixture rejection", "PASS"),
    ("Determinism", "4/4 PASS"),
    ("Day number validation", "5/5 PASS"),
    ("Golden - Partial", "PASS"),
    ("Golden - Selection limit", "PASS"),
    ("Golden - Normal", "PASS"),
    ("No-interests", "PASS"),
    ("D=4, E=7", "PASS"),
    ("D=4, E=2", "PASS"),
    ("D=1", "PASS"),
    ("D=14", "PASS"),
    ("E > 3D", "PASS"),
    ("Invalid days", "PASS"),
    ("Malformed candidates", "4/4 PASS"),
    ("Destination mismatch", "PASS"),
    ("Duplicate candidate IDs", "PASS"),
    ("Duplicate interests", "PASS"),
    ("Invalid coordinate ranges", "4/4 PASS"),
    ("Overlap case", "PASS"),
    ("Tie case", "PASS"),
]

print("=== Planner Independent Evaluation Summary ===")
print()

for check, result in evaluation_summary:
    print(f"{check:<35} {result}")

print()
print("Overall result: PASS")

=== Planner Independent Evaluation Summary ===

Valid fixtures                      5/5 PASS
INVALID fixture rejection           PASS
Determinism                         4/4 PASS
Day number validation               5/5 PASS
Golden - Partial                    PASS
Golden - Selection limit            PASS
Golden - Normal                     PASS
No-interests                        PASS
D=4, E=7                            PASS
D=4, E=2                            PASS
D=1                                 PASS
D=14                                PASS
E > 3D                              PASS
Invalid days                        PASS
Malformed candidates                4/4 PASS
Destination mismatch                PASS
Duplicate candidate IDs             PASS
Duplicate interests                 PASS
Invalid coordinate ranges           4/4 PASS
Overlap case                        PASS
Tie case                            PASS

Overall result: PASS


## Evaluation Conclusion

The real `planning.plan_trip()` implementation was evaluated independently
using contract-valid synthetic inputs and negative controls.

The evaluation covered:

- Capacity distribution and day structure
- Empty and partial itineraries
- Selection limit when `E > 3D`
- Minimum and maximum supported trip durations (`D=1` and `D=14`)
- No-interests behavior
- Deterministic repeated execution
- Golden/reference cases
- Invalid days
- Malformed candidates
- Destination mismatch
- Duplicate candidate IDs
- Duplicate interests
- Invalid and non-finite coordinates
- Overlap and tie cases
- Deliberately invalid planner output

All implemented checks passed, including rejection of the deliberately
invalid fixture.

The evaluator checks the planner contract and independent invariants only.
It does not reproduce the planner's internal ranking, geographic grouping,
or ordering logic.

# Stage 2A Evidence — Backend / FastAPI Integration

This section records the independent QA verification performed for Stage 2A.

The verification covers:

- Real FastAPI planning service availability.
- Real `plan_trip()` integration through `FastApiPlanningClient`.
- Normal, partial, and empty planning scenarios.
- Public Backend endpoint behavior.
- Independent verification of the Backend controller wiring.

QA does not modify or reimplement planner logic.

## 1. Real FastAPI Integration Test

The real FastAPI service was started at:

`http://127.0.0.1:8000`

The Backend integration tests were executed using:

```text
dotnet test "Backend\TripPlanning.Api.Tests\TripPlanning.Api.Tests.csproj" --filter "FullyQualifiedName~FastApiRealIntegrationTests" --logger "console;verbosity=normal"

Test results:

- Total tests: 3
- Passed: 3
- Failed: 0
- Skipped: 0

Verified scenarios:

- Normal planning result
- Partial itinerary warning
- Empty planning result

The three integration tests successfully communicated with the real FastAPI planning service.

## 2. Public Backend Endpoint Verification

The public Backend endpoint was tested while the real FastAPI service was running.

Endpoint:

`POST http://localhost:5185/trip-plans/preview`

Test scenario:

- Destination: `rome`
- Requested days: `3`
- Candidates: `rom-001`, `rom-002`

Observed response:

- HTTP status: `200 OK`
- 3 days returned
- `rom-001` assigned to Day 1
- `rom-002` assigned to Day 2
- Day 3 is empty
- Warning code: `PARTIAL_ITINERARY`
- `X-Request-ID` was returned by the Backend

This confirms that the public Backend endpoint is reachable and returns a structurally valid planning response for the tested case.

In [43]:
public_backend_response = {
    "status_code": 200,
    "day_numbers": [1, 2, 3],
    "selected_ids": ["rom-001", "rom-002"],
    "empty_days": [3],
    "warning_codes": ["PARTIAL_ITINERARY"],
    "request_id_present": True,
}

assert public_backend_response["status_code"] == 200
assert public_backend_response["day_numbers"] == [1, 2, 3]
assert public_backend_response["selected_ids"] == ["rom-001", "rom-002"]
assert public_backend_response["empty_days"] == [3]
assert "PARTIAL_ITINERARY" in public_backend_response["warning_codes"]
assert public_backend_response["request_id_present"] is True

print("Public Backend endpoint response checks: PASSED")

Public Backend endpoint response checks: PASSED


## 3. Initial Public Endpoint Wiring Verification — Before `b75b806`

The public Backend controller was independently inspected to verify which planning service is used by the endpoint.

Observed controller dependency:

`IFakeTripPreviewService`

Observed controller call:

`_tripPreviewService.GeneratePreview(request)`

Therefore, the public `POST /trip-plans/preview` endpoint is currently wired to the Fake Trip Preview service rather than the real `IPlanningService`.

As a result, the successful public endpoint response cannot be used as evidence that the following end-to-end path is active:

`Backend → IPlanningService → FastApiPlanningClient → FastAPI → plan_trip()`

No Backend or FastAPI implementation changes were made during QA verification.

## 4. Initial Stage 2A QA Status — Before `b75b806`

### Verified

- Independent planner checker is ready.
- Valid and invalid reusable cases are ready.
- Negative control is proven.
- Real FastAPI service was started successfully.
- Real FastAPI integration tests passed: 3/3.
- Normal, partial, and empty planning scenarios passed through `FastApiPlanningClient`.
- Public Backend endpoint is reachable and returns a structurally valid response.
- Backend `X-Request-ID` was observed.

### Blocking Evidence

The public `POST /trip-plans/preview` controller is still wired to:

`IFakeTripPreviewService`

rather than:

`IPlanningService`

Therefore, the public Backend → real FastAPI → `plan_trip()` path has not been independently verified.

### QA Conclusion

**Stage 2A: NOT GREEN YET**

The real FastAPI client integration is passing, but the public Backend endpoint is not currently proven to use the real planning service.

No Backend or FastAPI implementation changes were made by QA.

In [47]:
stage2a_request = {
    "destinationId": "rome",
    "days": 3,
    "interests": [],
    "candidatePlaces": [
        {
            "id": "rom-001",
            "destinationId": "rome",
            "name": "Colosseum",
            "categoryIds": ["historic_site", "monument"],
            "latitude": 41.8902,
            "longitude": 12.4922,
        },
        {
            "id": "rom-002",
            "destinationId": "rome",
            "name": "Roman Forum",
            "categoryIds": ["historic_site"],
            "latitude": 41.8925,
            "longitude": 12.4853,
        },
    ],
}

# Actual public Backend response from commit b75b806,
# normalized to the internal PlanningResult shape expected
# by the independent evaluator.

stage2a_backend_response = {
    "destinationId": "rome",
    "days": [
        {
            "dayNumber": 1,
            "places": [{"id": "rom-001"}],
        },
        {
            "dayNumber": 2,
            "places": [{"id": "rom-002"}],
        },
        {
            "dayNumber": 3,
            "places": [],
        },
    ],
    "warnings": [
        {
            "code": "PARTIAL_ITINERARY",
            "message": (
                "The current candidate pool does not contain enough "
                "places to cover every requested day."
            ),
        }
    ],
}

stage2a_response = {
    "destinationId": stage2a_backend_response["destinationId"],
    "days": [
        {
            "day": day["dayNumber"],
            "placeIds": [place["id"] for place in day["places"]],
        }
        for day in stage2a_backend_response["days"]
    ],
    "warnings": stage2a_backend_response["warnings"],
}

errors = []
errors.extend(
    validate_planning_output(stage2a_request, stage2a_response)
)
errors.extend(
    validate_warning_behavior(stage2a_request, stage2a_response)
)

if errors:
    print("Integrated Stage 2A independent checks: FAILED")
    for error in errors:
        print("-", error)
else:
    print("Integrated Stage 2A independent checks: PASSED")

Integrated Stage 2A independent checks: PASSED


## 5. Integrated Stage 2A Re-Verification — `b75b806`

Stage 2A was independently re-verified after the public Backend wiring fix in commit `b75b806`.

### Independent Runtime Evidence

The public Backend endpoint was exercised through:

`POST http://localhost:5185/trip-plans/preview`

The tested request used:

- Destination: `rome`
- Days: `3`
- Candidate places: `rom-001`, `rom-002`
- Interests: none

The public Backend returned:

- HTTP `200 OK`
- `X-Request-ID` present
- 3 requested days
- `rom-001` on Day 1
- `rom-002` on Day 2
- Day 3 empty
- `PARTIAL_ITINERARY` warning

Backend runtime logs independently showed an outgoing request to:

`POST http://127.0.0.1:8000/plan`

The FastAPI runtime log independently showed:

`POST /plan HTTP/1.1 200 OK`

The same integrated output was normalized from the public Backend DTO to the internal `PlanningResult` representation and passed through the existing independent evaluator.

Result:

**Integrated Stage 2A independent checks: PASSED**

### Regression Evidence

The complete Backend test suite was also executed on `b75b806`.

- Total tests: 25
- Passed: 25
- Failed: 0
- Skipped: 0

The suite includes coverage for controller wiring, request ID propagation, cancellation handling, malformed responses, unavailable FastAPI, timeouts, server failures, invalid planning results, and real FastAPI normal/partial/empty scenarios.

### QA Conclusion

The previously identified public-controller wiring blocker is no longer present on commit `b75b806`.

Independent QA verification found no remaining Stage 2A blocker in the tested integration path:

`Public Backend → IRealTripPreviewService → IPlanningService → FastApiPlanningClient → FastAPI /plan → plan_trip()`

The Stage 2A QA verification criteria are satisfied for commit `b75b806`.

Final Stage 2A GREEN / gate confirmation remains with the responsible project gate owner or reviewer.